In [ ]:
import pandas as pd
import os
import time
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

# --- 1. KONFIGURASI ---
CATALOG_PATH = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/katalog_2001_keatas.csv'
SAVE_DIR = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/waveform_data_2001'
LOG_FILE = os.path.join(SAVE_DIR, 'download_progress.txt') # File untuk mencatat progres

client = Client("IRIS")

def get_last_processed_row():
    """Membaca baris terakhir yang berhasil diunduh dari file log."""
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r') as f:
            lines = f.readlines()
            if lines:
                return int(lines[-1].strip())
    return 0

def download_micro_batch(df, num_batches=10, size_per_batch=10):
    # Tentukan mulai dari baris mana berdasarkan log
    start_row = get_last_processed_row()
    print(f"[INFO] Melanjutkan unduhan dari baris: {start_row}")
    
    current_index = start_row
    
    for i in range(num_batches):
        print(f"\n--- Memproses Micro-Batch {i+1}/{num_batches} (Baris: {current_index} ke {current_index + size_per_batch}) ---")
        
        bulk_list = []
        df_subset = df.iloc[current_index : current_index + size_per_batch].copy()
        
        for idx, row in df_subset.iterrows():
            try:
                # Konversi waktu ke format ISO untuk ObsPy
                ts = pd.to_datetime(f"{row['Date']} {row['Time (UTC)']}", dayfirst=True)
                t = UTCDateTime(ts.strftime('%Y-%m-%dT%H:%M:%S'))
                bulk_list.append(("*", "*", "*", "BHZ", t, t + 10))
            except:
                continue

        if bulk_list:
            try:
                # Eksekusi unduh
                st = client.get_waveforms_bulk(bulk_list)
                file_name = f"micro_batch_{current_index}.mseed"
                output_path = os.path.join(SAVE_DIR, file_name)
                
                st.write(output_path, format="MSEED")
                print(f"[SUKSES] Berhasil mengunduh batch baris {current_index}")
                
                # CATAT PROGRES: Simpan baris terakhir yang sukses
                current_index += size_per_batch
                with open(LOG_FILE, 'a') as f:
                    f.write(f"{current_index}\n")
                    
            except Exception as e:
                print(f"[ERROR] Gagal di baris {current_index}. Pesan: {e}")
                # Jika error 413 tetap muncul, kita hentikan agar tidak looping error
                if "413" in str(e):
                    print("[STOP] Server masih menolak ukuran ini. Coba perkecil size_per_batch menjadi 5.")
                    break
        
        time.sleep(5) # Jeda lebih lama untuk keamanan server

# --- EKSEKUSI ---
if os.path.exists(CATALOG_PATH):
    df_all = pd.read_csv(CATALOG_PATH)
    # Kita coba 10 batch kecil (Total 100 event)
    download_micro_batch(df_all, num_batches=10, size_per_batch=10)

In [ ]:
import pandas as pd
import os
import time
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

# --- 1. KONFIGURASI ---
CATALOG_PATH = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/katalog_2001_keatas.csv'
SAVE_DIR = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/waveform_data_2001'
LOG_FILE = os.path.join(SAVE_DIR, 'download_progress.txt')

client = Client("IRIS")

# Pastikan folder ada
os.makedirs(SAVE_DIR, exist_ok=True)

def get_last_processed_row():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r') as f:
            lines = f.readlines()
            if lines: return int(lines[-1].strip())
    return 0

def download_micro_batch_v5(df, num_batches=10, size_per_batch=10):
    start_row = get_last_processed_row()
    print(f"[INFO] Melanjutkan unduhan dari baris: {start_row}")
    
    current_index = start_row
    
    for i in range(num_batches):
        if current_index >= len(df): break
        
        print(f"\n--- Memproses Batch {i+1}/{num_batches} (Indeks: {current_index}) ---")
        
        bulk_list = []
        df_subset = df.iloc[current_index : current_index + size_per_batch].copy()
        
        for idx, row in df_subset.iterrows():
            try:
                # PENALARAN: Mengonversi format "01-Jan-2001 09:12:26"
                # dayfirst=True membantu pandas mengenali tanggal di awal
                datetime_str = f"{row['Date']} {row['Time (UTC)']}"
                ts = pd.to_datetime(datetime_str)
                
                # Konversi ke format ObsPy UTCDateTime
                t = UTCDateTime(ts.isoformat())
                
                # Tambahkan ke daftar bulk (Network, Station, Loc, Chan, Start, End)
                bulk_list.append(("*", "*", "*", "BHZ", t, t + 10))
            except Exception as e:
                print(f"[SKIP] Gagal parsing di baris {idx}: {e}")
                continue

        if bulk_list:
            try:
                # Eksekusi Bulk Download
                st = client.get_waveforms_bulk(bulk_list)
                
                file_name = f"micro_batch_{current_index}.mseed"
                output_path = os.path.join(SAVE_DIR, file_name)
                st.write(output_path, format="MSEED")
                
                print(f"[SUKSES] Berhasil menyimpan: {file_name}")
                
                # Catat progres
                current_index += size_per_batch
                with open(LOG_FILE, 'a') as f:
                    f.write(f"{current_index}\n")
                    
            except Exception as e:
                print(f"[ERROR] Batch {current_index} gagal: {e}")
                if "413" in str(e): break
        
        time.sleep(3) # Jeda untuk keamanan server

# --- EKSEKUSI ---
if os.path.exists(CATALOG_PATH):
    # Membaca CSV (Pastikan separator sesuai, biasanya ',' atau ';')
    df_all = pd.read_csv(CATALOG_PATH)
    download_micro_batch_v5(df_all, num_batches=10, size_per_batch=10)

In [ ]:
import pandas as pd
import os
import time
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

# --- 1. KONFIGURASI PATH ---
CATALOG_PATH = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/katalog_2001_keatas.csv'
BASE_SAVE_DIR = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/waveform_raw_massive'
client = Client("IRIS")

def download_massive_data(df, start_year=2001, end_year=2024, batch_size=10):
    # Konversi kolom Date ke datetime agar mudah difilter
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
    
    for year in range(start_year, end_year + 1):
        print(f"\n==== MEMPROSES TAHUN {year} ====")
        
        # Buat folder khusus per tahun
        year_dir = os.path.join(BASE_SAVE_DIR, str(year))
        os.makedirs(year_dir, exist_ok=True)
        
        # Filter data hanya untuk tahun tersebut
        df_year = df[df['Date'].dt.year == year]
        print(f"Ditemukan {len(df_year)} kejadian gempa.")
        
        # Proses per batch (size_per_batch)
        for start_idx in range(0, len(df_year), batch_size):
            subset = df_year.iloc[start_idx : start_idx + batch_size]
            bulk_list = []
            
            for idx, row in subset.iterrows():
                try:
                    # Gabungkan Tanggal dan Waktu
                    t_str = f"{row['Date'].strftime('%Y-%m-%d')} {row['Time (UTC)']}"
                    t = UTCDateTime(t_str)
                    # Ambil durasi 10 detik
                    bulk_list.append(("*", "*", "*", "BHZ", t, t + 10))
                except:
                    continue
            
            if bulk_list:
                try:
                    st = client.get_waveforms_bulk(bulk_list)
                    file_name = f"batch_{year}_{start_idx}.mseed"
                    st.write(os.path.join(year_dir, file_name), format="MSEED")
                    print(f"[OK] Batch {start_idx} tahun {year} selesai.")
                except Exception as e:
                    print(f"[ERROR] Batch {start_idx} gagal: {e}")
            
            time.sleep(2) # Jeda agar server IRIS tidak overload

# --- EKSEKUSI ---
if os.path.exists(CATALOG_PATH):
    df_full = pd.read_csv(CATALOG_PATH)
    # Jalankan proses unduhan masif
    download_massive_data(df_full, start_year=2001, end_year=2024, batch_size=10)